In [ ]:
import numpy as np
import numpy.linalg as npl

def gradientMethod(A, b, x0, kmax, relres_t):
    """
    Solves the linear system Ax = b using the Gradient Method (Steepest Descent).

    This method is intended for systems where A is symmetric and positive-definite (SPD).

    Parameters:
    A (np.ndarray): The coefficient matrix (should be SPD).
    b (np.ndarray): The right-hand side vector.
    x0 (np.ndarray): The initial guess for the solution.
    kmax (int): The maximum number of iterations.
    relres_t (float): The relative residual tolerance for convergence.

    Returns:
    np.ndarray: The approximate solution xk.
    """
    xk = x0
    rk = b - A @ xk  # Initial residual r_0 = b - A*x_0
    
    norm_b = npl.norm(b)
    
    # Handle the case b=0 to avoid division by zero
    if norm_b == 0:
        return np.zeros_like(b) # Solution is x=0
        
    relres = npl.norm(rk) / norm_b
    
    k = 0 # Initialize iteration counter
    
    # Iterate until max iterations or tolerance is met
    while k < kmax and relres > relres_t:
        
        # matrix-vector product A * r_k
        # This is needed for the step size and the residual update
        Ark = A @ rk
        
        # optimal step size (alpha_k)
        # alpha_k = (r_k^T * r_k) / (r_k^T * A * r_k)
        rk_dot_rk = rk @ rk  # Dot product r_k^T * r_k
        rk_dot_Ark = rk @ Ark # Dot product r_k^T * A * r_k
        
        # Check for non-positive-definiteness.
        # If rk is not zero, rk_dot_Ark should be positive for an SPD matrix.
        if rk_dot_Ark <= 0:
            print(f"Warning: Matrix may not be positive-definite. Stopping at iteration {k}.")
            break
            
        alpha_k = rk_dot_rk / rk_dot_Ark
        
        # 3. update solution
        # x_{k+1} = x_k + alpha_k * r_k
        xk = xk + alpha_k * rk
        
        # 4. update residual
        # r_{k+1} = r_k - alpha_k * (A * r_k)
        rk = rk - alpha_k * Ark
        
        # update relative residual and iteration count
        relres = npl.norm(rk) / norm_b
        k += 1
        
    # Return the final solution
    # Note: Corrected 'sol' to 'xk'
    return xk

### Third ex


(1000, 1000) (1000, 1000)
k₂(A₁) = 2.999980300323666
k₂(A₁) = 2.9999803003236662
k₂(A₂) = 406095.042664486


In [ ]:
import numpy as np
import numpy.linalg as npl
from scipy.sparse import issparse


def cg_linsys(A, b, x0, kmax, relres_t):
    """
    Solves the linear system Ax = b using the Gradient Method (Steepest Descent).

    This method is intended for systems where A is symmetric and positive-definite (SPD).

    Parameters:
    A (np.ndarray): The coefficient matrix (should be SPD).
    b (np.ndarray): The right-hand side vector.
    x0 (np.ndarray): The initial guess for the solution.
    kmax (int): The maximum number of iterations.
    relres_t (float): The relative residual tolerance for convergence.

    Returns:
    np.ndarray: The approximate solution xk.
    """
    xk = x0
    rk = b - A @ xk
    pk = rk
    k = 0
    
        # Convert sparse matrices to dense arrays and flatten if needed
    if issparse(b):
        b = b.toarray().flatten()
    else:
        b = np.asarray(b).flatten()
    
    if issparse(x0):
        x0 = x0.toarray().flatten()
    else:
        x0 = np.asarray(x0).flatten()
        
    norm_b = npl.norm(b)
    if norm_b == 0:
        return np.zeros_like(b)
    relres = npl.norm(rk) / norm_b
    while k < kmax and relres > relres_t:
        zk = A @ pk
        alpha_k = (rk.T @ pk) / (pk.T @ zk)
        xk = xk + alpha_k * pk
        rk = rk - alpha_k * zk
        beta_k = -(rk.T @ zk) / (pk.T @ zk)
        pk = rk + beta_k * pk
        relres = npl.norm(rk) / norm_b
        k += 1
    return xk

In [ ]:
import numpy as np
import numpy.linalg as npl
from scipy.sparse import issparse

def cg_linsys(A, b, x0, kmax, relres_t):
    """
    Solves the linear system Ax = b using the Conjugate Gradient Method.

    This method is intended for systems where A is symmetric and positive-definite (SPD).

    Parameters:
    A (np.ndarray or sparse matrix): The coefficient matrix (should be SPD).
    b (np.ndarray or sparse matrix): The right-hand side vector.
    x0 (np.ndarray): The initial guess for the solution.
    kmax (int): The maximum number of iterations.
    relres_t (float): The relative residual tolerance for convergence.

    Returns:
    np.ndarray: The approximate solution xk.
    """
    if issparse(b):
        b = b.toarray().flatten()
    else:
        b = np.asarray(b).flatten()
    if issparse(x0):
        x0 = x0.toarray().flatten()
    else:
        x0 = np.asarray(x0).flatten()
    xk = x0.copy()
    rk = b - A @ xk
    if issparse(rk):
        rk = rk.toarray().flatten()
    pk = rk.copy()
    k = 0
    norm_b = npl.norm(b)
    if norm_b == 0:
        return np.zeros_like(b)
    relres = npl.norm(rk) / norm_b
    while k < kmax and relres > relres_t:
        zk = A @ pk
        if issparse(zk):
            zk = zk.toarray().flatten()
        alpha_k = (rk.T @ pk) / (pk.T @ zk)
        xk = xk + alpha_k * pk
        rk = rk - alpha_k * zk
        beta_k = -(rk.T @ zk) / (pk.T @ zk)
        pk = rk + beta_k * pk
        relres = npl.norm(rk) / norm_b
        k += 1
    return xk

In [58]:
import scipy.io as sio

mat_contents = sio.loadmat('lab03_sparse_linsys.mat')

print(mat_contents.keys())

A = mat_contents['A']
b = mat_contents['b']
x0 = mat_contents['x0']
x_sol_exact = mat_contents.get('x_sol')

print(f"Matrix A: {A}")
print(f"Matrix A shape: {A.shape}")
print(f"Vector b shape: {b.shape}")


kmax = 1000
relres_t = 1e-14

final_solution_x = cg_linsys(A, b, x0, kmax, relres_t)
print(f"Solution computed, shape: {final_solution_x.shape}")

if x_sol_exact is not None:
            x_sol_exact = x_sol_exact.ravel()
            error = npl.norm(final_solution_x - x_sol_exact)
            print(f"Error (norm(x - x_exact)): {error:.2e}")



dict_keys(['__header__', '__version__', '__globals__', 'A', 'b', 'n', 'x0', 'x_sol'])
Matrix A: <COOrdinate sparse matrix of dtype 'float64'
	with 298 stored elements and shape (100, 100)>
  Coords	Values
  (0, 0)	4.0
  (1, 0)	-1.0
  (0, 1)	-1.0
  (1, 1)	4.0
  (2, 1)	-1.0
  (1, 2)	-1.0
  (2, 2)	4.0
  (3, 2)	-1.0
  (2, 3)	-1.0
  (3, 3)	4.0
  (4, 3)	-1.0
  (3, 4)	-1.0
  (4, 4)	4.0
  (5, 4)	-1.0
  (4, 5)	-1.0
  (5, 5)	4.0
  (6, 5)	-1.0
  (5, 6)	-1.0
  (6, 6)	4.0
  (7, 6)	-1.0
  (6, 7)	-1.0
  (7, 7)	4.0
  (8, 7)	-1.0
  (7, 8)	-1.0
  (8, 8)	4.0
  :	:
  (91, 91)	4.0
  (92, 91)	-1.0
  (91, 92)	-1.0
  (92, 92)	4.0
  (93, 92)	-1.0
  (92, 93)	-1.0
  (93, 93)	4.0
  (94, 93)	-1.0
  (93, 94)	-1.0
  (94, 94)	4.0
  (95, 94)	-1.0
  (94, 95)	-1.0
  (95, 95)	4.0
  (96, 95)	-1.0
  (95, 96)	-1.0
  (96, 96)	4.0
  (97, 96)	-1.0
  (96, 97)	-1.0
  (97, 97)	4.0
  (98, 97)	-1.0
  (97, 98)	-1.0
  (98, 98)	4.0
  (99, 98)	-1.0
  (98, 99)	-1.0
  (99, 99)	4.0
Matrix A shape: (100, 100)
Vector b shape: (100, 1)
Solut

## Exercise 3.3

In [59]:
n = 1000

# Diagonal values
alpha1 = 4
alpha2 = 2

# Create the main diagonals
main_diag_A1 = np.full(n, alpha1)
main_diag_A2 = np.full(n, alpha2)

# Create the upper and lower diagonals
off_diag = np.full(n - 1, -1)

# Build the tridiagonal matrices
A1 = np.diag(main_diag_A1) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1)
A2 = np.diag(main_diag_A2) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1)

print(A1.shape, A2.shape)

cond_A1 = np.linalg.cond(A1, 2)
cond_A1_2 = np.linalg.norm(A1, 2) * np.linalg.norm(np.linalg.inv(A1), 2)
cond_A2 = np.linalg.cond(A2, 2)

print("k₂(A₁) =", cond_A1)
print("k₂(A₁) =", cond_A1_2)
print("k₂(A₂) =", cond_A2)



(1000, 1000) (1000, 1000)
k₂(A₁) = 2.999980300323666
k₂(A₁) = 2.9999803003236662
k₂(A₂) = 406095.042664486
